# Too Hard / Soft 분석

In [109]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [111]:
data1 = pd.read_csv("review-vc_sales_by_collection.csv")
data2 = pd.read_csv("review-vc_sales_exclude_too_hard_too_soft.csv")
data3 = pd.read_csv("count_by_collection.csv")

In [160]:
data1 = data1[data1['collection'] != '__TOTAL__']
data2 = data2[data2['collection'] != '__TOTAL__']

In [164]:
data2

,yr_month,financial_category,collection,written_avg_rating,written_12_cnt,witten_all_cnt,written_12_ratio,all_avg_rating,all_12_cnt,all_all_cnt,all_12_ratio,sales_amount,sales_qty
1,202201,Box Springs,Walter 9in,3.500000,3.0,8.0,0.375000,NaN,NaN,NaN,NaN,91988.82,446.0
2,202201,Box Springs,Walter 7.5in,4.500000,1.0,10.0,0.100000,NaN,NaN,NaN,NaN,271300.75,1670.0
3,202201,Box Springs,Walter 4in,4.666667,1.0,9.0,0.111111,NaN,NaN,NaN,NaN,229874.12,1231.0
4,202201,Box Springs,Victor 9in,5.000000,0.0,2.0,0.000000,NaN,NaN,NaN,NaN,60917.50,297.0
5,202201,Box Springs,Victor 7.5in,4.750000,1.0,12.0,0.083333,NaN,NaN,NaN,NaN,57457.99,295.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
23388,202503,Toppers,1.5in GTFT w WonderBox,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,4938.07,123.0
23389,202503,Toppers,1.5in GT MF,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,2949.68,84.0
23390,202503,Toppers,1.5in Copper Convoluted,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,NaN,NaN
23391,202503,Toppers,1.25in Swirl Copper,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,67.94,1.0


### Data 차이 분석

In [166]:
avg_rating1 = data1.groupby(['financial_category','collection']).agg(
    avg_rating_before=('written_avg_rating', 'mean'),
    total_sales=('sales_amount', 'sum'),
    total_qty=('sales_qty','sum')
).reset_index()

In [168]:
avg_rating1

,financial_category,collection,avg_rating_before,total_sales,total_qty
0,Box Springs,5in OOT Box Spring,3.123065,212189.73,2463.0
1,Box Springs,9in OOT Box Spring,3.623485,250590.40,2460.0
2,Box Springs,Adrianne,NaN,913.93,15.0
3,Box Springs,Annemarie,4.257992,8114259.42,101235.0
4,Box Springs,Armita 5in,3.838275,17228795.75,136805.0
...,...,...,...,...,...
670,Toppers,4in PRMF,NaN,30008.31,320.0
671,Toppers,4in PRMF w Cover,5.000000,0.00,0.0
672,Toppers,4in SWFT w WonderBox,2.791667,168236.40,2104.0
673,Toppers,4in Swirl Gel,3.681159,442631.27,6061.0


In [170]:
#avg_rating1 = data1.groupby('collection')['written_avg_rating'].mean().rename('avg_rating_before')
#total_sales = data1.groupby('collection')['sales_amount'].sum().rename('total_sales')
avg_rating2 = data2.groupby('collection')['written_avg_rating'].mean().rename('avg_rating_after')
avg_diff_df = pd.merge(avg_rating1, avg_rating2, on='collection', how='left')
#avg_diff_df = pd.merge(avg_diff_df, total_sales, on='collection', how='left')
avg_diff_df = pd.merge(avg_diff_df, data3, on='collection', how='left')

In [172]:
avg_diff_df

,financial_category,collection,avg_rating_before,total_sales,total_qty,avg_rating_after,count
0,Box Springs,5in OOT Box Spring,3.123065,212189.73,2463.0,3.081399,NaN
1,Box Springs,9in OOT Box Spring,3.623485,250590.40,2460.0,3.623485,NaN
2,Box Springs,Adrianne,NaN,913.93,15.0,NaN,NaN
3,Box Springs,Annemarie,4.257992,8114259.42,101235.0,4.257992,NaN
4,Box Springs,Armita 5in,3.838275,17228795.75,136805.0,3.870545,NaN
...,...,...,...,...,...,...,...
670,Toppers,4in PRMF,NaN,30008.31,320.0,NaN,NaN
671,Toppers,4in PRMF w Cover,5.000000,0.00,0.0,5.000000,1.0
672,Toppers,4in SWFT w WonderBox,2.791667,168236.40,2104.0,2.791667,NaN
673,Toppers,4in Swirl Gel,3.681159,442631.27,6061.0,3.710145,6.0


In [174]:
avg_diff_df['rating_diff']=(avg_diff_df['avg_rating_before'] - avg_diff_df['avg_rating_after']).abs()
avg_diff_sorted = avg_diff_df.sort_values(by='total_sales', ascending=False)

In [176]:
avg_diff_sorted

,financial_category,collection,avg_rating_before,total_sales,total_qty,avg_rating_after,count,rating_diff
79,Foam Mattresses,12in Green Tea MF,3.163846,1.410941e+08,414720.0,3.545239,1226.0,0.381393
6,Box Springs,Armita 9in,3.773505,4.735485e+07,366797.0,3.778784,2.0,0.005279
78,Foam Mattresses,12in Gel Green Tea MF,3.306554,4.714150e+07,138592.0,3.646500,463.0,0.339946
110,Foam Mattresses,6in Green Tea MF,3.507009,4.613743e+07,352237.0,3.789563,209.0,0.282554
56,Foam Mattresses,10in Green Tea MF,3.312290,4.202334e+07,160861.0,3.704156,242.0,0.391866
...,...,...,...,...,...,...,...,...
150,Non Bedroom Furniture,Becky Side Table,NaN,0.000000e+00,0.0,NaN,NaN,NaN
674,Toppers,4in TorsoTec Copper,NaN,0.000000e+00,0.0,NaN,NaN,NaN
161,Non Bedroom Furniture,Console Table,NaN,-7.099000e+01,-1.0,NaN,NaN,NaN
173,Non Bedroom Furniture,Jamie Coffee Table,NaN,-9.099000e+01,-1.0,NaN,NaN,NaN


In [180]:
avg_diff_sorted.to_csv('avg_diff_sorted_out2.csv', index=False)